In [26]:
import pandas as pd


In [27]:
df_raw = pd.read_csv('export_api_influencers.csv')
df_raw.head()

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,Bot_5,India,2023-05-06,AMATEUR_1
1,fan_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,Bot_11,India,2023-02-14,AMATEUR_1
3,Bot_1,India,2023-11-24,AMATEUR_1
4,Bot_36,India,2023-06-24,AMATEUR_1


#### Fase 1: Data Engineering (La Explosión)

Para poder usar networkx, primero necesitan una Lista de Aristas purificada. Abran un nuevo Jupyter Notebook, importen pandas y realicen la siguiente tubería de datos (Data Pipeline):

1. Limpieza Estructural: Eliminen las filas donde la columna Cuentas_Que_Sigue esté vacía (nula). Estandaricen la columna Usuario a MAYÚSCULAS y quítenle los espacios al inicio/final. Eliminen usuarios duplicados.
2. La Explosión: Separen los nombres de la columna Cuentas_Que_Sigue usando .str.split(','). Luego, utilicen la función mágica de Pandas .explode() sobre esa columna para aplanar la tabla y convertirla en una verdadera lista de Origen -> Destino.
3. Purificación Post-Explosión: Vuelvan a limpiar los textos de la columna destino (mayúsculas y sin espacios extra). Finalmente, eliminen los Self-Loops (Registros donde el Usuario aparezca siguiéndose a sí mismo).

In [34]:
# Eliminar nulos
df = df_raw.dropna(subset=["Cuentas_Que_Sigue"])

# Quitar espacios y estarndarizar a Mayusculas
df["Usuario"] = df["Usuario"].str.strip().str.upper()

# Eliminar duplicates
df = df.drop_duplicates(subset=["Usuario"], keep="first")
df.head()


/var/folders/db/dln2q0qj2hj1087pbnh_yqmw0000gn/T/ipykernel_38689/3869156483.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Usuario"] = df["Usuario"].str.strip().str.upper()


,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,BOT_11,India,2023-02-14,AMATEUR_1
3,BOT_1,India,2023-11-24,AMATEUR_1
4,BOT_36,India,2023-06-24,AMATEUR_1


In [35]:
# Separar nombres por coma
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(",")

# Aplanar, una fila por cada cuenta seguida
df = df.explode("Cuentas_Que_Sigue")

# Renombrar a origen -> destino
df = df.rename(columns={"Usuario": "Origen", "Cuentas_Que_Sigue": "Destino"})

# Reiniciar indice
df = df.reset_index(drop=True)

df.head(10)


,Origen,Ubicacion,Fecha_Creacion,Destino
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,GARNACHO
2,FAN_6,Mexico,2023-07-14,MESSI
3,BOT_11,India,2023-02-14,AMATEUR_1
4,BOT_1,India,2023-11-24,AMATEUR_1
5,BOT_36,India,2023-06-24,AMATEUR_1
6,FAN_4,Mexico,2023-05-29,GARNACHO
7,BOT_29,India,2023-08-18,AMATEUR_1
8,BOT_13,India,2023-08-05,AMATEUR_1
9,BOT_18,India,2023-04-30,AMATEUR_1


In [ ]:
# Limpiar la columna Destino: quitar espacios y pasar a MAYÚSCULAS
df["Destino"] = df["Destino"].str.strip().str.upper()

# Eliminar self-loops (Origen igual a Destino)
df = df[df["Origen"] != df["Destino"]]

# Reiniciar el indice
df = df.reset_index(drop=True)

### Fase 2: El Engaño de la Popularidad

Con su Lista de Aristas terminada:  

1. Importen networkx y conviertan su tabla en un Grafo Dirigido (nx.DiGraph).
2. Calculen la Centralidad de Entrada (nx.in_degree_centrality).
3. Impriman a los 3 usuarios con el puntaje más alto.
- Pregunta: A simple vista, ¿quién parece ser el rey de la red?

In [ ]:
import networkx as nx

# Convertir la tabla en un grafo dirigido
G = nx.from_pandas_edgelist(df, source="Origen", target="Destino", create_using=nx.DiGraph)

# Centralidad de entrada
centralidad = nx.in_degree_centrality(G)

# Top 3 usuarios con el puntaje más alto
top3 = sorted(centralidad.items(), key=lambda x: x[1], reverse=True)[:3]

for usuario, puntaje in top3:
    print(f"{usuario}: {puntaje:.4f}")

AMATEUR_1: 0.7231
GARNACHO: 0.2000
MESSI: 0.1538


¿A simple vista, ¿quién parece ser el rey de la red?

- Amateur_1 es quien se encuentra al principio de la lista con un puntaje de 0.7, es decri, el 70% de la red lo siguen.

### Fase 3: La Verdad Topológica (Las Métricas) 

Es hora de usar los algoritmos que vimos en los videos de hoy para auditar la red.

1. Calculen el PageRank (nx.pagerank). ¿Quién tiene ahora la verdadera autoridad?
2. Conviertan el grafo a No Dirigido y calculen el Coeficiente de Clustering (nx.clustering). Impriman el valor exacto de AMATEUR_1 y el de GARNACHO.

In [38]:
# PageRank (sobre el grafo dirigido)
pr = nx.pagerank(G)
top_pr = pd.Series(pr).nlargest(5)
print("Top 5 PageRank:")
print(top_pr)

# Pasar a no dirigido y calcular clustering
G_und = G.to_undirected()
clustering = nx.clustering(G_und)

print("\nClustering AMATEUR_1:", clustering["AMATEUR_1"])
print("Clustering GARNACHO:", clustering["GARNACHO"])

Top 5 PageRank:
AMATEUR_1    0.196941
DE_PAUL      0.088285
GARNACHO     0.084288
SUAREZ       0.079382
OTAMENDI     0.068620
dtype: float64

Clustering AMATEUR_1: 0
Clustering GARNACHO: 0.13333333333333333


- Amateur_1 sigue siendo quien se posiciona al inicio de la lista


### Veredicto Final (Markdown)


Veredicto Final (Markdown): Cierren su notebook con una celda de texto. Usando estrictamente sus resultados matemáticos de PageRank y Clustering, argumenten de forma técnica y profesional:

- ¿Quién compró bots y por qué?
- ¿Quién es el influencer real y por qué?

Resultados obtenidos

| Usuario   | PageRank | Clustering (no dirigido) |
|-----------|----------|--------------------------|
| AMATEUR_1 | 0.1969   | 0.7231                   |
| GARNACHO  | 0.0843   | 0.2000                   |
| MESSI     | n/d (fuera del Top 5) | 0.1538      |


¿Quién compro bots?

- AMATEUR_1 es el principal sospechoso por estas dos métricas:

1. PageRank dominante (0.1969):concentra casi el 20% de la autoridad de toda la red y más que duplica al segundo lugar (DE_PAUL, 0.0883). Una brecha tan grande sugiere una acumulación de seguidores poco natural.
2. Clustering extremo (0.7231): el 72% de los vecinos de AMATEUR_1 están conectados entre sí. En una audiencia orgánica, los seguidores de una cuenta suelen ser personas independientes que no se siguen unas a otras. Un clustering tan alto indica una estructura cerrada y densa, típica de granjas de bots que se siguen mutuamente para simular legitimidad.

Es decir, su autoridad proviene de un círculo interconectado y no de una audiencia diversa.


¿Quién es el influencer real y por qué?
- GARNACHO es el candidato con perfil orgánico

1. PageRank alto y consistente (0.0843, 3.er lugar): tiene autoridad real dentro de la red, comparable a la de las demás cuentas del Top 5.
2. Clustering bajo (0.2000): sus vecinos están poco conectados entre sí, lo que indica una audiencia diversa y dispersa, no un grupo cerrado. Este es el patrón esperado en una cuenta que atrae seguidores de comunidades distintas.

A diferencia de AMATEUR_1, su influencia no depende de una estructura cerrada. Es autoridad construida con conexiones independientes.